# 09 · Keyword Classifier Validation (Phase 4b/Step 3)

What this notebook covers: the automated (no-human-labels-required) validation checks for the
BM25F keyword classifier (`src/classify_by_keywords.py`) that replaces BERTopic as the canonical
topic-to-document link — per the topic-model-redo plan's "Validation" section. This is a report on
work already done, not a driver of it: it reads already-computed artifacts
(`data/processed/topic_keyword_assignments.parquet`, `data/processed/topic_assignments.parquet`,
`outputs/topic_keywords.json`, `outputs/topic_labels.json`) plus one live re-run of the classifier
with `--tiebreak embedding` for the embedding-centroid diagnostic. Light deps only
(pandas/numpy/matplotlib — no bertopic/torch/umap/hdbscan); executed end-to-end during authoring
to verify its own numbers, same convention as notebook 08.

**What this notebook does NOT do: report real accuracy.** The stratified gold set
(`data/gold/topic_gold_set.csv`, built by `src/build_gold_sample.py`) exists but has **not been
human-labeled yet** — every `human_parent_label` cell is empty. Accuracy-by-`conf_tier` (the
plan's actual calibration test) does not exist as a number anywhere in this repo. Section 5 below
says this plainly; nothing here should be read as calibration evidence.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'data' / 'processed').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

import sys
sys.path.insert(0, str(REPO_ROOT))

PROC = REPO_ROOT / 'data' / 'processed'
OUTPUTS = REPO_ROOT / 'outputs'
GOLD = REPO_ROOT / 'data' / 'gold'

from src.classify_by_keywords import ARTIFACT_TOPIC_ID, load_curated_taxonomy
from src.validate_keyword_classifier import (
    title_only_normalization_report, parent_level_bertopic_agreement,
    embedding_centroid_report, gold_set_report,
)

kw = pd.read_parquet(PROC / 'topic_keyword_assignments.parquet')
kw['doc_id'] = kw['doc_id'].astype(str)
ta = pd.read_parquet(PROC / 'topic_assignments.parquet')
ta['doc_id'] = ta['doc_id'].astype(str)
gr = pd.read_parquet(PROC / 'grants.parquet')
gr['grant_id'] = gr['grant_id'].astype(str)

print(f'topic_keyword_assignments.parquet: {len(kw)} rows')
print(f'topic_assignments.parquet (BERTopic, comparison column): {len(ta)} rows')

## 1 · Coverage headline: before (BERTopic) vs. after (keyword classifier)

The number this whole redesign targets: "Unassigned" (no confident topic) grants/dollars, per
`CLAUDE.md`'s own caveat #5 (697 grants / $583M / 26.7% of dollars under BERTopic, as of the
2026-08-20 refit).

In [ ]:
grants_dollars = gr.set_index('grant_id')['totaldollars']

# BEFORE — BERTopic: noise (-1) or the ARTIFACT_TOPIC_ID placeholder-title cluster.
ta_grants = ta[~ta['is_extra']].copy()
before_unassigned_ids = set(ta_grants.loc[ta_grants['topic_id'].isin([-1, ARTIFACT_TOPIC_ID]), 'doc_id'])
before_n = len(before_unassigned_ids)
before_dollars = grants_dollars.reindex(list(before_unassigned_ids)).fillna(0).sum()
before_pct_n = before_n / len(ta_grants)

# AFTER — keyword classifier: kw_leaf_id == -1.
kw_grants = kw[~kw['is_extra']].copy()
after_unassigned = kw_grants[kw_grants['kw_leaf_id'] == -1]
after_n = len(after_unassigned)
after_dollars = grants_dollars.reindex(after_unassigned['doc_id']).fillna(0).sum()
after_pct_n = after_n / len(kw_grants)

print(f"BEFORE (BERTopic):          {before_n} grants ({before_pct_n:.1%}), ${before_dollars/1e6:,.1f}M")
print(f"AFTER  (keyword classifier): {after_n} grants ({after_pct_n:.1%}), ${after_dollars/1e6:,.1f}M")
print()
print('AFTER by reason:')
print(after_unassigned['unassigned_reason'].value_counts().to_string())
print()
print("Note: 'before' figures recomputed live here from topic_assignments.parquet, so may differ "
      "slightly from CLAUDE.md's snapshot if the corpus changed since; treat CLAUDE.md's number as "
      "the historical headline and this cell's as the live comparison basis for 'after'.")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(['BERTopic\n(before)', 'Keyword classifier\n(after)'],
       [before_pct_n * 100, after_pct_n * 100], color=['#c7ccd3', '#4C78A8'])
ax.set_ylabel('% of grants unassigned')
ax.set_title('Unassigned rate: before vs. after')
for i, v in enumerate([before_pct_n * 100, after_pct_n * 100]):
    ax.text(i, v + 0.5, f'{v:.1f}%', ha='center')
plt.tight_layout()
plt.show()

## 2 · Title-only normalization check

The plan's own "most important automatic test," because it's a direct falsifiable test of the
BM25F length-normalization term. Split every metric by `modelTitleOnly` (the same field
`build_viz_data.py` defines: did the topic model see any usable abstract text). Expected: the
low/none confidence rate for title-only docs should be within ~10pp of abstract-bearing docs, and
mean margin should NOT be higher for title-only (higher would mean the `W_TITLE` weight
over-boosts).

In [ ]:
tno = title_only_normalization_report(kw)

rows = []
for key in ('abstract_bearing', 'title_only'):
    r = tno[key]
    rows.append({'group': key, 'n': r['n'], 'unassigned_rate': r['unassigned_rate'],
                 'low_or_none_rate': r['low_or_none_rate'],
                 'mean_margin_rel_assigned': r['mean_margin_rel_assigned']})
tno_df = pd.DataFrame(rows).set_index('group')
display(tno_df.style.format({'unassigned_rate': '{:.1%}', 'low_or_none_rate': '{:.1%}',
                              'mean_margin_rel_assigned': '{:.3f}'}))

print(f"\nlow+none rate gap: {tno['_summary']['low_or_none_rate_gap_pp']} pp "
      f"(within 10pp target: {tno['_summary']['within_10pp']})")
print(f"mean margin higher for title-only (over-boost signature): "
      f"{tno['_summary']['mean_margin_higher_for_title_only']}")

**Result: this check FAILS both criteria as currently calibrated.** The low/none rate gap is
12.2pp (target: within ~10pp), and mean `margin_rel` for assigned title-only docs (0.685) is
*higher*, not lower, than for abstract-bearing docs (0.373) — the exact over-boost failure
signature the plan names. This is a real, currently-uncalibrated limitation, not a passing result
dressed up: `K1`/`B`/`ALPHA`/`W_TITLE` in `classify_by_keywords.py` are explicitly documented there
as literature-standard placeholders pending gold-set calibration (Section 5), not tuned against
this corpus. A title-only doc that matches even one curated term currently gets an
outsized-looking margin because its very short length means one match dominates the score — this
is exactly the kind of miscalibration a real gold set should catch and let `W_TITLE`/`K1`/`B` be
retuned against, not something to paper over here.

## 3 · Parent-level BERTopic agreement

Bands, written down before running (per the plan): **<0.55** → the keyword lists don't describe
this corpus, stop and re-curate; **0.70–0.90** → healthy; **>0.97** → the scorer has just
re-derived BERTopic's own clustering, bought inspectability but no new information.

Methodology note: this is a majority-vote crosswalk computed **per old BERTopic topic_id** (not a
hand-authored mapping between the two taxonomies' differently-named parent labels, which doesn't
exist and would be a subjective judgment call) — for each BERTopic topic, the majority
`kw_parent_id` among its own docs; a doc "agrees" if its own `kw_parent_id` matches that majority.
This is the same methodology `classify_by_keywords.py`'s own `_attach_bertopic_columns` already
uses at leaf granularity, redone here at parent granularity (7 classes, not 31) — the actual
"parent-level" comparison the plan calls for.

In [ ]:
agree = parent_level_bertopic_agreement(kw)
print(f"grain: {agree['grain']}")
print(f"overall agreement rate: {agree['overall_agreement_rate']:.1%}")
print(f"band: {agree['band_interpretation']}")
display(agree['by_old_parent'].style.format({'agreement_rate': '{:.1%}'}))

**Result: 67.9%, in the "borderline" range** — below the 0.70 healthy floor but well above the
0.55 re-curate threshold. Read this as a genuine sanity-check pass with room to improve, not a
strong endorsement: the plan is explicit that this ceiling is inherently limited (the keyword
lists were curated by comparison against these same BERTopic clusters, so full agreement was never
the goal and >0.97 would itself be a bad sign). The by-old-parent breakdown shows real variation
(0.56–0.88) — "Life Sciences & Biomedicine" and "Society, Health & Mobility" pull the average down,
worth a closer look in any future re-curation pass; "Computing & Cybersecurity" and "Education &
Learning" already agree well.

## 4 · Embedding-centroid independent signal

A genuinely independent signal (disjoint information from the lexical BM25F scorer): does the
new label for a doc BERTopic left as noise/artifact-but-with-real-text actually sit near other
confidently-labeled docs in SPECTER2 embedding space? Two named failure signatures: margins
indistinguishable from confident docs (the confidence measure isn't measuring anything), or
concentrated in only 2-3 parents (a coverage hole in the same place HDBSCAN had one, just
relabeled). This re-runs the classifier once with `--tiebreak embedding` (~30-60s) — that flag is
diagnostic-only and never changes `kw_leaf_id`.

In [ ]:
leaves, parents = load_curated_taxonomy()
centroid = embedding_centroid_report(leaves, parents)

print(f"formerly-noise-with-text docs now assigned a real leaf: "
      f"{centroid['n_formerly_noise_with_text_now_assigned']}")
print(f"mean centroid margin, formerly-noise docs: {centroid['mean_centroid_margin_formerly_noise']:.4f}")
print(f"mean centroid margin, confident docs:      {centroid['mean_centroid_margin_confident']:.4f}")
print(f"plan's own reference (noise vs. BERTopic-assigned), context only: "
      f"{centroid['plan_reference_margins']['noise_docs_reference']} vs. "
      f"{centroid['plan_reference_margins']['bertopic_assigned_reference']}")
print()
print(f"concentration check: {centroid['concentration_flag']}")
print(f"margin check: {centroid['margin_indistinguishable_flag']}")
print()
print('parent distribution of formerly-noise-with-text docs:')
for k, v in sorted(centroid['formerly_noise_parent_distribution'].items(), key=lambda kv: -kv[1]):
    print(f'  {k}: {v:.1%}')

**Result: a genuinely encouraging, independent signal.** The formerly-noise-with-text docs'
mean centroid margin (~0.011) sits between the plan's own cited noise-population reference
(0.008) and confident-population reference (0.025) — closer to the noise end, as expected (these
are the hard cases), but clearly non-zero and measurably below confident docs' own margin
(~0.024), so the confidence gradient is *not* decorative — the second failure signature (margins
indistinguishable from confident docs) is **not** observed. The parent distribution is spread
across all 7 parents (no single parent above ~25%), so the third failure signature
(concentration in 2-3 parents, meaning a coverage hole just got relabeled rather than resolved) is
also **not** observed. This is the strongest evidence in this notebook that the new assignments
for BERTopic's former noise bucket carry real structure, not noise dressed up as confidence.

## 5 · Gold set — scaffold only, NOT yet labeled

**This is the only real ground truth check in the plan, and it has not been run.**
`src/build_gold_sample.py` draws a stratified n=180 sample (agency × text-availability ×
BERTopic-status, oversampling noise-with-text) to `data/gold/topic_gold_set.csv`, with model
predictions kept in a *separate* file (`topic_gold_set_predictions.csv`) specifically so a human
can label blind. As of this notebook's authoring, every `human_parent_label` cell in that CSV is
empty — **accuracy-by-`conf_tier` (the plan's actual calibration test) does not exist as a number
anywhere in this repo.** Nothing in Sections 1-4 above should be read as a substitute for it: they
are sanity checks and an independent signal, not accuracy against ground truth.

In [ ]:
gold_status = gold_set_report()
print(gold_status['status'])

if (GOLD / 'topic_gold_set.csv').exists():
    gold_df = pd.read_csv(GOLD / 'topic_gold_set.csv')
    print(f"\nsample composition ({len(gold_df)} rows) by stratum:")
    print(gold_df['stratum'].value_counts().to_string())

## 6 · Summary — honest takeaways

- **Coverage headline moved**, as designed: Unassigned dropped from BERTopic's baseline to the
  numbers in Section 1 (see that cell's live output — the by-reason breakdown there,
  `no_keyword_evidence` in particular, is itself a real curation-coverage gap worth a closer look,
  not something this redesign eliminated outright).
- **Title-only normalization currently FAILS** (Section 2) — the BM25F length term as currently
  configured over-boosts title-only docs' apparent confidence, not under-scores them as intended.
  This is a calibration problem (`K1`/`B`/`ALPHA`/`W_TITLE` are uncalibrated placeholders), not a
  design-of-formula problem, but it means `conf_tier` should not yet be trusted more for
  title-only docs than for abstract-bearing ones — if anything, currently, less.
- **BERTopic agreement is borderline-healthy** (Section 3, 67.9%) — a real sanity-check pass, with
  room to improve, and real per-old-parent variation worth a closer look.
- **The embedding-centroid signal is genuinely encouraging** (Section 4) — neither named failure
  signature (indistinguishable margins, parent concentration) was observed for the
  formerly-noise-with-text population.
- **No accuracy number exists yet** (Section 5). Before treating `conf_tier` as calibrated, or
  before any downstream integration decision that assumes this classifier is "better than
  BERTopic," the gold set needs to actually be labeled and scored — that is the next concrete
  piece of work this notebook identifies, not something it can shortcut.